# Inspect Ingested Data Tables

This notebook helps you inspect the SQLite bronze tables used by Skimmer.

- Default DB path: `data/skimmer.db`
- Override with env var: `SKIMMER_DB_PATH`
- It lists tables, shows schema, row counts, and recent records.

In [3]:
import json
import os
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

DEFAULT_DB_PATH = Path('data/skimmer.db')
DB_PATH = Path(os.environ.get('SKIMMER_DB_PATH', DEFAULT_DB_PATH)).expanduser()

print(f'Using database: {DB_PATH.resolve()}')
if not DB_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DB_PATH}')

Using database: /home/orangepi/code_projects/skimmer/data/skimmer.db


In [4]:
connection = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql_query(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

if tables_df.empty:
    raise RuntimeError('No tables found in the database.')

display(tables_df)

,table_name
0,bronze_socialblade_channel_stats
1,bronze_vidiq_channel_stats
2,bronze_youtube_skimmed
3,collection_attempts
4,collection_errors
5,profile_queue


In [5]:
def table_schema(table_name: str) -> pd.DataFrame:
    return pd.read_sql_query(f"PRAGMA table_info({table_name})", connection)


def row_count(table_name: str) -> int:
    query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    return int(pd.read_sql_query(query, connection).iloc[0]['row_count'])


def preview_table(table_name: str, limit: int = 25) -> pd.DataFrame:
    schema = table_schema(table_name)
    column_names = schema['name'].tolist()
    order_column = next(
        (
            column
            for column in ('id', 'last_seen_at', 'observed_at', 'occurred_at', 'created_at', 'latest_video_at')
            if column in column_names
        ),
        next((column for column, is_primary_key in zip(schema['name'], schema['pk']) if is_primary_key), None),
    )
    order_by = f' ORDER BY "{order_column}" DESC' if order_column else ''
    query = f'SELECT * FROM "{table_name}"{order_by} LIMIT {int(limit)}'
    df = pd.read_sql_query(query, connection)

    if 'raw_record_json' in df.columns:
        def parse_raw(value):
            if value is None:
                return None
            try:
                return json.loads(value)
            except (json.JSONDecodeError, TypeError):
                return value

        df['raw_record_json_parsed'] = df['raw_record_json'].map(parse_raw)

    return df


In [6]:
summary_rows = []
for table_name in tables_df['table_name'].tolist():
    summary_rows.append({'table_name': table_name, 'row_count': row_count(table_name)})

summary_df = pd.DataFrame(summary_rows).sort_values(['row_count', 'table_name'], ascending=[False, True])
display(summary_df)

,table_name,row_count
2,bronze_youtube_skimmed,10688
3,collection_attempts,4155
5,profile_queue,2094
1,bronze_vidiq_channel_stats,1513
0,bronze_socialblade_channel_stats,311
4,collection_errors,292


In [7]:
for table_name in tables_df['table_name'].tolist():
    print('\n' + '=' * 100)
    print(f'TABLE: {table_name}')
    print('=' * 100)

    print('\nSchema:')
    display(table_schema(table_name))

    print('\nLatest rows:')
    display(preview_table(table_name, limit=20))


TABLE: bronze_socialblade_channel_stats

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,subscribers_change,,0,None,0
3,3,subscribers_total,,0,None,0
4,4,views_change,,0,None,0
5,5,views_total,,0,None,0
6,6,videos_change,,0,None,0
7,7,videos_total,,0,None,0
8,8,earnings_low,,0,None,0
9,9,earnings_high,,0,None,0



Latest rows:


,id,channel_id,subscribers_change,subscribers_total,views_change,views_total,videos_change,videos_total,earnings_low,earnings_high,data_digest
0,311,UCp90hB8_sqLcEEXVgiUCbcw,1000.0,251000,11301,11248718,NaN,377,3,45,430b7ac14efa41886e1a53e8cc17b9192d0d2fb540dabf...
1,310,UCp90hB8_sqLcEEXVgiUCbcw,NaN,250000,14796,11237417,2.0,377,4,59,6929dab2336ccc2db81c7261cdd7d0b945f585f72db020...
2,309,UCp90hB8_sqLcEEXVgiUCbcw,NaN,250000,11370,11222621,NaN,375,3,45,6ccebe1db6649eec7fd114d929b677d1befb401d77d645...
3,308,UCp90hB8_sqLcEEXVgiUCbcw,NaN,250000,19442,11211251,NaN,375,5,78,1f799d2787a707e227bc45ae6fcc7d0ec952acb90fb1f1...
4,307,UCp90hB8_sqLcEEXVgiUCbcw,NaN,250000,14745,11191809,1.0,375,4,59,93ccf1df46865a2e90d7bce25037bc6729502168ba3c41...
5,306,UCp90hB8_sqLcEEXVgiUCbcw,NaN,250000,17594,11177064,1.0,374,4,70,252d86095d1d2515403d8274ef141f21f89aa93c242482...
6,305,UCp90hB8_sqLcEEXVgiUCbcw,1000.0,250000,20429,11159470,1.0,373,5,82,77cc635d570cf1c159b75639e1f4e084f4acbc6b7dae8a...
7,304,UCp90hB8_sqLcEEXVgiUCbcw,NaN,249000,23942,11139041,2.0,372,6,96,c2c1c3781f2e1886d8ad886a2c36fe2725edf705cf0515...
8,303,UCp90hB8_sqLcEEXVgiUCbcw,NaN,249000,11693,11115099,1.0,370,3,47,05fb8717032356a699c0571ab8023781ec84007f5c2ed8...
9,302,UCp90hB8_sqLcEEXVgiUCbcw,NaN,249000,14035,11103406,NaN,369,4,56,502d2ab81d38c90a273a9dda8910cb2da7d27e1876bbf9...



TABLE: bronze_vidiq_channel_stats

Schema:


...

,id,occurred_at,source,channel_id,source_url,error_type,status_code,message
0,292,2026-07-21T19:22:26+00:00,socialblade,UC7_YxT-KID8kRbqZo7MyscQ,https://socialblade.com/youtube/channel/UC7_Yx...,cloudflare_block,403.0,Cloudflare blocked the headed browser session.
1,291,2026-07-21T19:18:56+00:00,vidiq,@NonstopDan,https://vidiq.com/youtube-stats/channel/@Nonst...,metrics_unavailable,NaN,vidIQ metrics unavailable.
2,290,2026-07-21T18:58:56+00:00,vidiq,@PapaMeat,https://vidiq.com/youtube-stats/channel/@PapaMeat,metrics_unavailable,NaN,vidIQ metrics unavailable.
3,289,2026-07-21T18:58:23+00:00,vidiq,@Homeworthy,https://vidiq.com/youtube-stats/channel/@Homew...,metrics_unavailable,NaN,vidIQ metrics unavailable.
4,288,2026-07-21T18:47:54+00:00,vidiq,@Fox29Philly,https://vidiq.com/youtube-stats/channel/@Fox29...,metrics_unavailable,NaN,vidIQ metrics unavailable.
5,287,2026-07-21T18:46:49+00:00,vidiq,@ESPNFC,https://vidiq.com/youtube-stats/channel/@ESPNFC,metrics_unavailable,NaN,vidIQ metrics unavailable.
6,286,2026-07-21T18:38:22+00:00,vidiq,@PearlsideChurch,https://vidiq.com/youtube-stats/channel/@Pearl...,metrics_unavailable,NaN,vidIQ metrics unavailable.
7,285,2026-07-21T18:36:08+00:00,socialblade,UCX-BbEqgGi6z0Nf1mYYR6bw,https://socialblade.com/youtube/channel/UCX-Bb...,cloudflare_block,403.0,Cloudflare blocked the headed browser session.
8,284,2026-07-21T18:35:38+00:00,vidiq,@yankees,https://vidiq.com/youtube-stats/channel/@yankees,metrics_unavailable,NaN,vidIQ metrics unavailable.
9,283,2026-07-21T18:33:24+00:00,vidiq,@Whoseplatesmacks,https://vidiq.com/youtube-stats/channel/@Whose...,metrics_unavailable,NaN,vidIQ metrics unavailable.



TABLE: profile_queue

Schema:


,cid,name,type,notnull,dflt_value,pk
0,0,channel_key,TEXT,0,None,1
1,1,channel_id,TEXT,1,None,0
2,2,channel_name,TEXT,0,None,0
3,3,latest_video_at,TEXT,1,None,0
4,4,last_seen_at,TEXT,1,None,0
5,5,last_success_at,TEXT,0,None,0
6,6,digested,INTEGER,1,0,0
7,7,assigned_source,TEXT,1,None,0
8,8,vidiq_failed,INTEGER,1,0,0
9,9,socialblade_failed,INTEGER,1,0,0



Latest rows:


,channel_key,channel_id,channel_name,latest_video_at,last_seen_at,last_success_at,digested,assigned_source,vidiq_failed,socialblade_failed,needs_review,claimed_by,claimed_at,youtube_channel_id,channel_id_claimed_by,channel_id_claimed_at,youtube_channel_id_attempted_at
0,@abcnews,@ABCNews,ABC News,2026-07-21T15:03:00+00:00,2026-07-21T19:18:37+00:00,2026-07-20T10:34:41+00:00,1,vidiq,1,1,1,None,None,None,None,None,None
1,@airrack,@airrack,Airrack,2026-07-17T07:34:11+00:00,2026-07-21T19:18:37+00:00,2026-07-20T12:12:27+00:00,0,vidiq,0,1,0,None,None,None,None,None,None
2,@alexandragater,@AlexandraGater,Alexandra Gater,2026-07-15T07:34:12+00:00,2026-07-21T19:18:37+00:00,2026-07-20T12:35:30+00:00,0,vidiq,0,0,0,None,None,None,None,None,None
3,@ampexclusive,@AMPEXCLUSIVE,AMP,2026-07-17T07:34:11+00:00,2026-07-21T19:18:37+00:00,2026-07-20T12:12:43+00:00,0,vidiq,0,1,0,None,None,None,None,None,None
4,@andreijikh,@AndreiJikh,Andrei Jikh,2026-07-20T19:18:37+00:00,2026-07-21T19:18:37+00:00,None,1,vidiq,1,1,1,None,None,None,None,None,None
5,@animeaddicts33,@AnimeAddicts33,AnimeAddicts,2026-07-04T07:34:15+00:00,2026-07-21T19:18:37+00:00,2026-07-20T02:39:11+00:00,1,vidiq,0,0,0,None,None,None,None,None,None
6,@asmontv,@AsmonTV,Asmongold TV,2026-07-21T17:18:37+00:00,2026-07-21T19:18:37+00:00,2026-07-21T03:01:29+00:00,0,socialblade,1,0,0,None,None,UCQeRaTukNYft1_6AZPACnog,None,None,2026-07-21T03:57:55+00:00
7,@baseballbatbros,@baseballbatbros,The Baseball Bat Bros,2026-07-17T18:34:11+00:00,2026-07-21T19:18:37+00:00,None,1,socialblade,1,1,1,None,None,None,None,None,None
8,@blueyofficialchannel,@BlueyOfficialChannel,Bluey - Official Channel,2026-07-21T01:02:54+00:00,2026-07-21T19:18:37+00:00,None,1,socialblade,1,1,1,None,None,None,None,None,None
9,@brainwaveworkspace,@BrainwaveWorkspace,Brainwave Workspace,2026-04-19T07:34:14+00:00,2026-07-21T19:18:37+00:00,2026-07-20T13:18:54+00:00,1,vidiq,0,1,0,None,None,None,None,None,None


In [ ]:
connection.close()